# Dataset de posiciones nominales (DEL/DEF/MED...)

Este notebook construye un dataset supervisado para clasificar el **rol nominal** del jugador (según acta), usando tracking + homografía ya generados.

Pipeline implementado:
1. Cargar tracks proyectados al campo 2D (`x,y` normalizados).
2. Generar plantilla JSON de etiquetado por frame cada 6 segundos (`0, 6, 12, ...`).
3. Rellenar `role_label` por `team_id + player_id + frame_id` en ese JSON.
4. Aplicar etiquetas al dataframe de observaciones.
5. Reorientar coordenadas por equipo objetivo (ataque hacia `+x`).
6. Construir features de jugador objetivo + compañeros (padding/máscara).
7. Exportar dataset por ejecución y añadirlo al dataset común acumulado.


In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "config.yaml").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró config.yaml subiendo desde este notebook.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiments.positions.position_dataset import (
    ROLE_LABELS_V1,
    add_velocity_features,
    append_run_to_common_dataset,
    apply_periodic_role_labels,
    build_observations_from_tracks,
    build_periodic_label_schedule,
    build_periodic_role_label_template,
    build_role_samples,
    find_project_root,
    get_video_fps,
    infer_attack_direction_by_team,
    list_position_videos,
    load_role_label_entries_json,
    load_tracks_json,
    render_frame_with_player_ids,
    resolve_tracks_path_for_video,
    save_role_label_template_json,
    validate_periodic_role_labels,
)

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
PROJECT_ROOT


In [ ]:
# Configuración base
VIDEOS = list_position_videos(PROJECT_ROOT)
if not VIDEOS:
    raise FileNotFoundError("No se encontraron vídeos en data/partidosPosiciones")

# Elige un vídeo de la lista
VIDEO_PATH = VIDEOS[0]

# Por defecto busca output/tracks_json/tracker/<nombre_video>_tracks.json
# Si no existe, cae a output/tracks_json/tracker/tracks.json (legacy).
TRACKS_PATH = resolve_tracks_path_for_video(PROJECT_ROOT, VIDEO_PATH)

MATCH_ID = VIDEO_PATH.stem.replace(" ", "_")
FIELD_LENGTH_M = 106.0
FIELD_WIDTH_M = 68.0
MAX_TEAMMATES = 10

# Etiquetado temporal periódico
LABEL_EVERY_SECONDS = 6.0
LABEL_START_SECONDS = 0.0
LABEL_MIN_PLAYERS = 18
LABELS_DIR = PROJECT_ROOT / "output" / "datasets" / "positions" / "labels"
LABELS_JSON_PATH = LABELS_DIR / f"{MATCH_ID}_labels_every_{int(LABEL_EVERY_SECONDS)}s.json"

print("VIDEO_PATH:", VIDEO_PATH)
print("TRACKS_PATH:", TRACKS_PATH)
print("MATCH_ID:", MATCH_ID)
print("LABELS_JSON_PATH:", LABELS_JSON_PATH)


In [ ]:
# Carga y tabla base (Fase 2 + Fase 3 base)
tracks = load_tracks_json(TRACKS_PATH)

obs_df = build_observations_from_tracks(
    tracks=tracks,
    match_id=MATCH_ID,
    field_length_m=FIELD_LENGTH_M,
    field_width_m=FIELD_WIDTH_M,
    tracked_classes=("player", "goalkeeper"),
)
obs_df = add_velocity_features(obs_df)

if obs_df.empty:
    raise ValueError("No hay observaciones player/goalkeeper con field_position_m en tracks.json")

print("Observaciones:", len(obs_df))
print("Frames con jugadores:", obs_df['frame_id'].nunique())
print("Equipos:", sorted(obs_df['team_id'].unique().tolist()))

display(obs_df.head(10))

In [ ]:
# Muestreo de frames de etiquetado cada 6 segundos (0, 6, 12, ...)
VIDEO_FPS = get_video_fps(VIDEO_PATH, default_fps=25.0)

label_schedule_df = build_periodic_label_schedule(
    observations=obs_df,
    fps=VIDEO_FPS,
    every_seconds=LABEL_EVERY_SECONDS,
    start_seconds=LABEL_START_SECONDS,
    min_players=LABEL_MIN_PLAYERS,
)
if label_schedule_df.empty:
    raise ValueError("No se pudo construir una planificación de etiquetado temporal.")

label_template = build_periodic_role_label_template(
    observations=obs_df,
    match_id=MATCH_ID,
    video_path=VIDEO_PATH,
    fps=VIDEO_FPS,
    every_seconds=LABEL_EVERY_SECONDS,
    start_seconds=LABEL_START_SECONDS,
    min_players=LABEL_MIN_PLAYERS,
)

created_template = not LABELS_JSON_PATH.exists()
save_role_label_template_json(label_template, LABELS_JSON_PATH, overwrite=False)

print(f"FPS detectado: {VIDEO_FPS:.3f}")
print("Frames de etiquetado planificados:", len(label_schedule_df))
display(label_schedule_df.head(10))

if created_template:
    print("Plantilla JSON creada en:", LABELS_JSON_PATH)
else:
    print("Plantilla JSON existente (no se sobrescribe):", LABELS_JSON_PATH)

print("Rellena role_label en el JSON y vuelve a ejecutar desde la celda de carga de etiquetas.")

preview_frame_id = int(label_schedule_df.iloc[0]["frame_id"])
preview_players = obs_df[obs_df["frame_id"] == preview_frame_id].copy()
preview_players = preview_players.sort_values(["team_id", "player_id"])

preview_bgr = render_frame_with_player_ids(VIDEO_PATH, preview_players, preview_frame_id)
preview_rgb = cv2.cvtColor(preview_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(16, 9))
plt.imshow(preview_rgb)
plt.title(f"Primer frame de etiquetado: {preview_frame_id}")
plt.axis("off")
plt.show()

cols = ["team_id", "player_id", "class_name", "x", "y", "vx", "vy", "confidence_tracking"]
display(preview_players[cols].reset_index(drop=True))


## Etiquetado manual por JSON cada 6 segundos

Rellena `role_label` en `LABELS_JSON_PATH` para los `frame_id` generados cada 6 segundos.
Las etiquetas permitidas están en `ROLE_LABELS_V1`.


In [ ]:
ROLE_LABELS = list(ROLE_LABELS_V1)
ROLE_LABELS

In [ ]:
ROLE_LABELS = list(ROLE_LABELS_V1)

label_entries_df = load_role_label_entries_json(LABELS_JSON_PATH)
validation = validate_periodic_role_labels(obs_df, label_entries_df, allowed_labels=ROLE_LABELS)

print("Validación de etiquetas periódicas:")
print(validation)

display(label_entries_df.head(20))

if validation["invalid_labels"]:
    raise ValueError(f"Etiquetas inválidas detectadas: {validation['invalid_labels']}")


In [ ]:
# Aplica etiquetas del JSON a observaciones (solo en frames etiquetados)
labeled_obs_df = apply_periodic_role_labels(obs_df, label_entries_df)
coverage = labeled_obs_df['role_label'].notna().mean()

print(f"Cobertura etiquetada por fila: {coverage:.2%}")
print("Filas etiquetadas:", int(labeled_obs_df['role_label'].notna().sum()))
print("Frames etiquetados:", int(labeled_obs_df.loc[labeled_obs_df['role_label'].notna(), 'frame_id'].nunique()))

display(labeled_obs_df[labeled_obs_df['role_label'].notna()].head(10))


## Orientación por equipo objetivo (Fase 4)

Convención aplicada por muestra:
- equipo objetivo siempre ataca hacia `+x`
- `y=0` banda izquierda del equipo objetivo
- si un equipo ataca hacia la izquierda, se aplica `x' = 1 - x`

In [ ]:
attack_direction_by_team, attack_details = infer_attack_direction_by_team(labeled_obs_df)
print("Dirección de ataque inferida (+1 derecha, -1 izquierda):")
print(attack_direction_by_team)
display(attack_details)

# Si quieres forzar manualmente:
# attack_direction_by_team = {"Real Madrid": +1, "Wolfsburgo": -1}

In [ ]:
# Construcción de muestras (Fase 5, 6 y 7)
if labeled_obs_df['role_label'].notna().sum() == 0:
    raise ValueError("No hay etiquetas en role_label. Rellena LABELS_JSON_PATH y vuelve a ejecutar.")

samples_df, teammates_tensor, teammate_mask, feature_spec = build_role_samples(
    observations_with_roles=labeled_obs_df,
    attack_direction_by_team=attack_direction_by_team,
    max_teammates=MAX_TEAMMATES,
    drop_unlabeled=True,
)

print("N muestras:", len(samples_df))
print("Forma teammates_tensor:", teammates_tensor.shape)
print("Forma teammate_mask:", teammate_mask.shape)
print("Features objetivo:", feature_spec.objective_feature_names)
print("Features compañeros:", feature_spec.teammate_feature_names)

display(samples_df.head(10))
if not samples_df.empty:
    display(samples_df['label'].value_counts(dropna=False).rename('count').to_frame())


In [ ]:
# Exportación de dataset por ejecución + append al dataset común
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = PROJECT_ROOT / "output" / "datasets" / "positions" / f"{MATCH_ID}_{run_id}"
out_dir.mkdir(parents=True, exist_ok=True)

base_table_path = out_dir / "base_table.csv"
samples_csv_path = out_dir / "samples_metadata_and_obj_features.csv"
samples_npz_path = out_dir / "samples_teammates.npz"
meta_json_path = out_dir / "dataset_meta.json"

labeled_obs_df.to_csv(base_table_path, index=False)
samples_df.to_csv(samples_csv_path, index=False)
np.savez_compressed(
    samples_npz_path,
    teammates_tensor=teammates_tensor.astype(np.float32),
    teammate_mask=teammate_mask.astype(np.uint8),
)

meta = {
    "created_at": datetime.now().isoformat(),
    "match_id": MATCH_ID,
    "video_path": str(VIDEO_PATH),
    "tracks_path": str(TRACKS_PATH),
    "labels_json_path": str(LABELS_JSON_PATH),
    "label_every_seconds": float(LABEL_EVERY_SECONDS),
    "field_length_m": FIELD_LENGTH_M,
    "field_width_m": FIELD_WIDTH_M,
    "max_teammates": MAX_TEAMMATES,
    "role_labels": ROLE_LABELS,
    "objective_feature_names": list(feature_spec.objective_feature_names),
    "teammate_feature_names": list(feature_spec.teammate_feature_names),
    "attack_direction_by_team": {k: int(v) for k, v in attack_direction_by_team.items()},
    "num_base_rows": int(len(labeled_obs_df)),
    "num_labeled_rows": int(labeled_obs_df['role_label'].notna().sum()),
    "num_samples": int(len(samples_df)),
}

import json
with meta_json_path.open("w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

append_summary = append_run_to_common_dataset(
    project_root=PROJECT_ROOT,
    labeled_obs_df=labeled_obs_df,
    samples_df=samples_df,
    teammates_tensor=teammates_tensor,
    teammate_mask=teammate_mask,
    feature_spec=feature_spec,
    source_info={
        "run_id": run_id,
        "match_id": MATCH_ID,
        "video_path": str(VIDEO_PATH),
        "tracks_path": str(TRACKS_PATH),
        "labels_json_path": str(LABELS_JSON_PATH),
        "num_samples": int(len(samples_df)),
    },
)

print("Dataset guardado en:", out_dir)
print(" -", base_table_path.name)
print(" -", samples_csv_path.name)
print(" -", samples_npz_path.name)
print(" -", meta_json_path.name)

print("
Dataset común actualizado en:", append_summary["common_dir"])
print(" - total base rows:", append_summary["num_base_rows"])
print(" - total samples:", append_summary["num_samples"])
print(" -", append_summary["samples_csv_path"].name)
print(" -", append_summary["samples_npz_path"].name)
print(" -", append_summary["meta_json_path"].name)
